# ESPECIALIZACIÓN EN INTELIGENCIA ARTIFICIAL Y BIG DATA

---

| Campo | Información |
|---|---|
| **Módulo** | BIG DATA APLICADO |
| **Profesor** | Víctor J. González |
| **Unidad de Trabajo** | UT02. Almacenamiento distribuido e integridad con HDFS |
| **Práctica** | PR0204. Pipeline de ingesta y procesamiento en Data Lake con HDFS |
| **Resultados de aprendizaje** | RA3 |

---

# PR0204: Pipeline de ingesta y procesamiento en Data Lake con HDFS

## 1. Contexto

El equipo de Analítica Avanzada de movilidad necesita estudiar la infraestructura ferroviaria de España y el flujo de viajeros utilizando los datos abiertos publicados por **Renfe** en su portal oficial ([https://data.renfe.com/](https://data.renfe.com/)).

Como Ingeniero/a de Datos del proyecto, tu objetivo es desarrollar un script en Python (utilizando la librería `hdfs`) que automatice la gestión del **Data Lake corporativo en HDFS**.

Tu programa deberá encargarse de:

- Crear la arquitectura de directorios.
- Ingestar los conjuntos de datos crudos (*Raw Data*).
- Procesarlos mediante flujos de datos (*streaming*) para no comprometer la memoria RAM del servidor de ingesta.
- Extraer subconjuntos de datos provinciales.
- Mantener un registro de auditoría (*logs*) de las operaciones realizadas.

## 2. Obtención de los datos de origen

Antes de comenzar a programar el *pipeline*, accede al portal de datos abiertos de Renfe ([https://data.renfe.com/dataset?res_format=CSV](https://data.renfe.com/dataset?res_format=CSV)) y descarga en tu directorio de trabajo local los siguientes conjuntos de datos en formato **CSV**:

1. **Estaciones - Listado completo** (renómbralo localmente como `estaciones_completo.csv`).
2. **Estaciones de FEVE** (renómbralo localmente como `estaciones_feve.csv`).
3. **Volumen de viajeros por franja horaria (León AM)** o de cualquier otro núcleo disponible (renómbralo localmente como `viajeros_tramo.csv`).

In [15]:
!pip install hdfs


In [16]:
from pathlib import Path

DATA_DIR = Path("data")
if not DATA_DIR.exists():
    DATA_DIR = Path("ut02/pr0204/data")

ARCHIVOS_LOCALES = {
    "estaciones_completo": DATA_DIR / "estaciones_completo.csv",
    "estaciones_feve": DATA_DIR / "estaciones_feve.csv",
    "viajeros_tramo": DATA_DIR / "viajeros_tramo.csv",
}

for clave, ruta in ARCHIVOS_LOCALES.items():
    if ruta.exists():
        print(f"{clave}: {ruta} -> OK ({ruta.stat().st_size} bytes)")
    else:
        print(f"{clave}: {ruta} -> NO ENCONTRADO")


estaciones_completo: data\estaciones_completo.csv -> OK (142155 bytes)
estaciones_feve: data\estaciones_feve.csv -> OK (61795 bytes)
viajeros_tramo: data\viajeros_tramo.csv -> OK (28822 bytes)



## 3. Requerimientos técnicos del Pipeline en Python

Desarrolla un cuaderno de Jupyter que ejecute de forma secuencial las siguientes fases sobre el sistema de ficheros distribuido.

### Fase 1: Conexión y arquitectura del Data Lake

En los entornos profesionales de Big Data, los datos no se vuelcan en la raíz del sistema de ficheros, sino que se estructuran en capas.

1. Establece la conexión con el clúster HDFS.
2. Crea de forma recursiva la siguiente estructura de directorios para nuestro proyecto, asegurándote de aplicar permisos POSIX `755`:

   - `/datalake/renfe/raw/estaciones`: Zona para los datos crudos de estaciones.
   - `/datalake/renfe/raw/viajeros`: Zona para los datos crudos de afluencia.
   - `/datalake/renfe/processed`: Zona para los datos limpios y filtrados.
   - `/datalake/renfe/logs`: Zona para el registro de auditoría del sistema.

3. Verifica que las carpetas dentro de `/datalake/renfe` se han generado correctamente.

In [17]:
# Fase 1: conexión y arquitectura del Data Lake
import os

from hdfs import InsecureClient

HDFS_URL = os.environ.get("HDFS_URL", "http://localhost:9870")
HDFS_USER = os.environ.get("HDFS_USER", "root")

print(f"Conectando con WebHDFS en {HDFS_URL} como usuario {HDFS_USER}")
client = InsecureClient(HDFS_URL, user=HDFS_USER)
print(client)

BASE = "/datalake/renfe"
DIRECTORIOS = [
    f"{BASE}/raw/estaciones",
    f"{BASE}/raw/viajeros",
    f"{BASE}/processed",
    f"{BASE}/logs",
]

for ruta in DIRECTORIOS:
    client.makedirs(ruta, permission="755")
    client.set_permission(ruta, "755")
    print(f"Directorio listo: {ruta} (permisos 755)")

# Verificación de la estructura creada
print(f"\nContenido de {BASE}: {client.list(BASE)}")
for ruta in DIRECTORIOS:
    print(f"  {ruta} -> {client.list(ruta)}")


Conectando con WebHDFS en http://localhost:9870 como usuario root
<InsecureClient(url='http://localhost:9870')>
Directorio listo: /datalake/renfe/raw/estaciones (permisos 755)
Directorio listo: /datalake/renfe/raw/viajeros (permisos 755)
Directorio listo: /datalake/renfe/processed (permisos 755)
Directorio listo: /datalake/renfe/logs (permisos 755)

Contenido de /datalake/renfe: ['logs', 'processed', 'raw']
  /datalake/renfe/raw/estaciones -> ['estaciones_completo.csv']
  /datalake/renfe/raw/viajeros -> ['viajeros_tramo.csv']
  /datalake/renfe/processed -> ['estaciones_filtradas.csv']
  /datalake/renfe/logs -> ['ingesta.log']


### Fase 2: Ingesta concurrente de datos crudos (Zona *Raw*)

1. Sube los archivos `estaciones_completo.csv` y `estaciones_feve.csv` al directorio `/datalake/renfe/raw/estaciones`.
2. Sube el archivo `viajeros_tramo.csv` al directorio `/datalake/renfe/raw/viajeros`.
3. Todas las subidas deben configurarse para sobrescribir ficheros previos en caso de reejecución del *pipeline* (`overwrite=True`) y deben emplear **4 hilos concurrentes** (`n_threads=4`) para optimizar la transferencia.
4. Realiza una auditoría de los archivos recién subidos en `/datalake/renfe/raw/estaciones`, obteniendo también los metadatos de los archivos.

   Recorre el resultado con Python e imprime por pantalla únicamente:

   - El **nombre de cada archivo**.
   - Su **tamaño en bytes** (`length`).
   - Sus **permisos** (`permission`).
   - Su **factor de replicación** (`replication`).

In [ ]:
# Fase 2: ingesta concurrente en la zona Raw y auditoría de metadatos
client = InsecureClient("http://localhost:14000", user=HDFS_USER)
print(f"Cliente HttpFS para transferencias: {client}")
SUBIDAS = [
    (ARCHIVOS_LOCALES["estaciones_completo"], f"{BASE}/raw/estaciones"),
    (ARCHIVOS_LOCALES["estaciones_feve"], f"{BASE}/raw/estaciones"),
    (ARCHIVOS_LOCALES["viajeros_tramo"], f"{BASE}/raw/viajeros"),
]

for local, destino_hdfs in SUBIDAS:
    client.upload(
        hdfs_path=destino_hdfs,
        local_path=str(local),
        overwrite=True,
        n_threads=4,
    )
    print(f"Subido: {local.name} -> {destino_hdfs}")

# Auditoría: nombre, tamaño, permisos y factor de replicación
print(f"\nAuditoría de {BASE}/raw/estaciones:")
for nombre, metadata in client.list(f"{BASE}/raw/estaciones", status=True):
    print(
        f"Nombre: {nombre} | length: {metadata['length']} bytes | "
        f"permission: {metadata['permission']} | replication: {metadata['replication']}"
    )


Cliente HttpFS para transferencias: <InsecureClient(url='http://localhost:14000')>
Subido: estaciones_completo.csv -> /datalake/renfe/raw/estaciones
Subido: estaciones_feve.csv -> /datalake/renfe/raw/estaciones
Subido: viajeros_tramo.csv -> /datalake/renfe/raw/viajeros

Auditoría de /datalake/renfe/raw/estaciones:
Nombre: estaciones_completo.csv | length: 142155 bytes | permission: 755 | replication: 3
Nombre: estaciones_feve.csv | length: 61795 bytes | permission: 755 | replication: 3


### Fase 3: Procesamiento en *Streaming* y filtrado (Zona *Processed*)

El archivo `estaciones_completo.csv` contiene todas las estaciones de la red, pero el departamento de negocio necesita un fichero específico que contenga únicamente las estaciones pertenecientes a una provincia concreta (por ejemplo, `LEÓN`, `MADRID` o `ASTURIAS`).

Dado que en un entorno de producción estos ficheros pueden ocupar decenas de Gigabytes, **está prohibido cargar el fichero completo en memoria RAM** (no se permite usar Pandas ni `.readlines()` sobre todo el archivo).

1. Abre un flujo de lectura en *streaming* sobre `/datalake/renfe/raw/estaciones/estaciones_completo.csv` utilizando el método `.read()` (recuerda especificar la codificación adecuada, como `'utf-8'` o `'latin-1'` según venga el CSV de Renfe).
2. Procesa el archivo línea a línea:
   - Conserva la primera línea (cabeceras del CSV).
   - Filtra todas aquellas líneas que correspondan a la provincia elegida.
3. Utilizando el método `.write()` (con `overwrite=True`), guarda el resultado filtrado directamente en HDFS en la ruta `/datalake/renfe/processed/estaciones_filtradas.csv`.
4. Lee en *streaming* las primeras **5 líneas** del nuevo archivo `/datalake/renfe/processed/estaciones_filtradas.csv` e imprímelas por pantalla (usando `.strip()`) para comprobar que el proceso de transformación ha funcionado.

In [19]:
# Fase 3: filtrado en streaming por la columna PROVINCIA (sin cargar todo el CSV)
import csv

CODIFICACION = "cp1252"   # Codificación del CSV original de Renfe
PROVINCIA = "LEÓN"        # Valor de la columna PROVINCIA que se conserva

ruta_entrada = f"{BASE}/raw/estaciones/estaciones_completo.csv"
ruta_salida = f"{BASE}/processed/estaciones_filtradas.csv"

contadores = {"leidas": 0, "filtradas": 0}


def filtrar_por_provincia():
    """Generador: cede la cabecera y las filas cuya columna PROVINCIA coincide."""
    with client.read(ruta_entrada, encoding=CODIFICACION) as lector:
        yield next(lector)                      # cabecera, tal cual
        for linea in lector:
            contadores["leidas"] += 1
            campos = next(csv.reader([linea], delimiter=";"), [])
            # La provincia es la columna 7 (índice 7) del CSV de estaciones.
            if len(campos) > 7 and campos[7].strip().strip('"') == PROVINCIA:
                contadores["filtradas"] += 1
                yield linea


# `write()` acepta un generador: escribe línea a línea sin acumular el fichero en RAM.
client.write(
    ruta_salida,
    data=filtrar_por_provincia(),
    encoding=CODIFICACION,
    overwrite=True,
)

estaciones_leidas = contadores["leidas"]
estaciones_filtradas = contadores["filtradas"]

print(f"Provincia (columna PROVINCIA): {PROVINCIA}")
print(f"Codificación del original: {CODIFICACION}")
print(f"Estaciones leídas: {estaciones_leidas}")
print(f"Estaciones filtradas: {estaciones_filtradas}")
print(f"Fichero generado: {ruta_salida}")


Provincia (columna PROVINCIA): LEÓN
Codificación del original: cp1252
Estaciones leídas: 1033
Estaciones filtradas: 43
Fichero generado: /datalake/renfe/processed/estaciones_filtradas.csv


In [20]:
# Comprobación: primeras cinco líneas del resultado
print("Primeras cinco líneas del resultado:\n")
with client.read(ruta_salida, encoding=CODIFICACION) as lector:
    for numero, linea in enumerate(lector):
        if numero == 5:
            break
        print(linea.strip())


Primeras cinco líneas del resultado:

"CODIGO";"DESCRIPCION";"LATITUD";"LONGITUD";"DIRECION";"CP";"POBLACION";"PROVINCIA";"PAIS";"CERCANIAS";"ANCHO MÉTRICO";"COMUN"
"05755";"LA LLAMA DE LA GUZPEÑA";"42.7702481";"-5.0412478";"Ctra. Cl - 626 --- Estación Ferrocarril, S/N";"24893";"PRADO DE LA GUZPEÑA";"LEÓN";"ESPAÑA";"SI";"SI";""
"05756";"LA ESPINA";"42.7868560";"-4.9052025";"Ctra. Cl - 626 --- Estación Ferrocarril, S/N";"24889";"VALDERRUEDA";"LEÓN";"ESPAÑA";"SI";"SI";""
"05757";"VALCUENDE";"42.7655780";"-4.9292558";"C/ Arriba, S/N";"24889";"VALDERRUEDA";"LEÓN";"ESPAÑA";"SI";"SI";""
"05758";"PUENTE ALMUHEY";"42.7876320";"-4.9718992";"Ctra. General  S/N- Puente Almuhey";"24880";"VALDERRUEDA";"LEÓN";"ESPAÑA";"SI";"SI";""


### Fase 4: Registro de auditoría (*Logging* con `append`)

Todo proceso ETL o de ingeniería de datos debe dejar rastro de sus operaciones en un fichero de registro sin borrar los eventos anteriores.

1. Crea un archivo de registro en `/datalake/renfe/logs/ingesta.log` utilizando el método `.write()` con `overwrite=True`, escribiendo una primera línea con el formato:

   ```text
   [INICIO] Pipeline ejecutado. Archivos crudos cargados en /raw.
   ```

2. A continuación, simula el registro de un segundo evento tras finalizar el filtrado de la Fase 3.

   Utiliza de nuevo el método `.write()` sobre el mismo archivo `/datalake/renfe/logs/ingesta.log`, pero esta vez configurando el parámetro `append=True` para añadir al final del fichero la línea:

   ```text
   [PROCESADO] Se han filtrado un total de X estaciones para la provincia seleccionada.
   ```

   > Sustituye `X` por el número real de estaciones que tu código haya contabilizado en la Fase 3.

3. Muestra por pantalla el contenido completo de `ingesta.log` leyéndolo mediante `.read()`.

In [ ]:
# Fase 4: registro de auditoría con write (overwrite) + write (append)
ruta_log = f"{BASE}/logs/ingesta.log"

# Creación del log.
with client.write(ruta_log, encoding=CODIFICACION, overwrite=True) as escritor:
    escritor.write("[INICIO] Pipeline ejecutado. Archivos crudos cargados en /raw.\n")

# append=True conserva la línea anterior.
with client.write(ruta_log, encoding=CODIFICACION, append=True) as escritor:
    escritor.write(
        f"[PROCESADO] Se han filtrado un total de {estaciones_filtradas} "
        "estaciones para la provincia seleccionada.\n"
    )

# Mostramos el contenido completo mediante .read()
print(f"Contenido completo de {ruta_log}:\n")
with client.read(ruta_log, encoding=CODIFICACION) as lector:
    print(lector.read().strip())


Contenido completo de /datalake/renfe/logs/ingesta.log:

[INICIO] Pipeline ejecutado. Archivos crudos cargados en /raw.
[PROCESADO] Se han filtrado un total de 43 estaciones para la provincia seleccionada.


### Fase 5: Exportación para el equipo de analítica y limpieza (Data Lifecycle)

1. El equipo de ciencia de datos solicita una copia local del fichero ya limpio y procesado.

   Utiliza el método `.download()` para descargar `/datalake/renfe/processed/estaciones_filtradas.csv` a tu máquina local con el nombre `entrega_estaciones.csv` (habilitando `overwrite=True`).

2. Por políticas de ahorro de almacenamiento en el clúster, una vez procesados los datos o finalizadas las pruebas, debemos aplicar políticas de retención:

   - **Eliminación de un archivo concreto:**

     Elimina únicamente el archivo `/datalake/renfe/raw/estaciones/estaciones_feve.csv` utilizando el método `.delete()` y muestra el listado de esa carpeta para confirmar que `estaciones_completo.csv` permanece intacto.

   - **Función de reinicio del Data Lake:**

     Crea una función en tu código llamada `reset_datalake()` que permita eliminar toda la estructura `/datalake/renfe` de forma recursiva (`recursive=True`) en caso de querer reiniciar el entorno desde cero.

     Déjala definida y comentada al final del cuaderno, o ejecútala tras mostrar todos los resultados anteriores.

In [22]:
# Fase 5: entrega local, limpieza selectiva y función de reinicio
ruta_local = Path("entrega_estaciones.csv")
client.download(
    hdfs_path=f"{BASE}/processed/estaciones_filtradas.csv",
    local_path=str(ruta_local),
    overwrite=True,
)
print(f"Archivo descargado: {ruta_local.name} ({ruta_local.stat().st_size} bytes)")
with ruta_local.open(encoding=CODIFICACION, newline="") as f:
    filas_locales = sum(1 for _ in csv.reader(f, delimiter=";")) - 1
print(f"Estaciones en la copia local: {filas_locales}")

# Eliminamos solo el CSV de FEVE y confirmamos que el completo sigue ahí.
archivo_feve = f"{BASE}/raw/estaciones/estaciones_feve.csv"
client.delete(archivo_feve)
print(f"Archivo eliminado: {Path(archivo_feve).name}")
print(f"Archivos que quedan en {BASE}/raw/estaciones: {client.list(f'{BASE}/raw/estaciones')}")


def reset_datalake():
    """Elimina toda la estructura /datalake/renfe de forma recursiva."""
    client.delete(BASE, recursive=True)
    print(f"Data Lake {BASE} reiniciado (eliminado por completo).")


# Descomenta la siguiente línea cuando quieras volver al estado original:
# reset_datalake()


Archivo descargado: entrega_estaciones.csv (5825 bytes)
Estaciones en la copia local: 43
Archivo eliminado: estaciones_feve.csv
Archivos que quedan en /datalake/renfe/raw/estaciones: ['estaciones_completo.csv']
